# Base-rate merged results

Explore `data/base_rate/base_rate_merged_results.csv` from a benchmark run.

Each row has **`score`** (`true`/`false`): whether the parsed answer matches **`scepticism_score_target`**. Unparseable rows have `score=false` and `parseable=false`.

In [41]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "base_rate").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MERGED_DIR = ROOT / "data" / "base_rate"
MERGED_CSV = None
for name in (
    # "base_rate_merged_results.csv",
    "base_rate_merged_results (7).csv",
):
    candidate = MERGED_DIR / name
    if candidate.is_file():
        MERGED_CSV = candidate
        break
if MERGED_CSV is None:
    raise FileNotFoundError(
        f"Missing merged results under {MERGED_DIR}. Run the base-rate benchmark first "
        "(benchmark/base-rate-benchmark.ipynb)."
    )

df = pd.read_csv(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
elif "score_outcome" in df.columns:
    df["score_value"] = (df["score_outcome"] == "normative").astype(int)
else:
    raise KeyError("Merged CSV must include 'score' or legacy 'score_outcome'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")

df["score_true"] = df["score_value"].astype(bool)

print("Loaded:", MERGED_CSV)
print("Rows:", len(df))
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
df.head()

Loaded: c:\src2\sceptical-llms\data\base_rate\base_rate_merged_results (7).csv
Rows: 34
Models: ['google/gemini-2.5-flash']
Vignettes: 10


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,parsed_answer_type,parsed_percent,parsed_choice,parsed_confidence,scoring_type,parseable,score,score_value,parseable_bool,score_true
0,actor_waiter_overlap__overlap__implausible__mc_full_probs,actor waiter overlap,overlap,small,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,False,implausible,...,mc_choice,NaN,A,5,mc_full,True,False,0,True,False
1,actor_waiter_overlap__overlap__mc_full_probs,actor waiter overlap,overlap,small,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,False,underdetermined,...,mc_choice,NaN,A,5,mc_full,True,True,1,True,True
2,actor_waiter_overlap__overlap__mc_numeric_probs,actor waiter overlap,overlap,small,mc_numeric,True,mc_numeric_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,False,underdetermined,...,mc_choice,NaN,B,5,mc_numeric,True,False,0,True,False
3,actor_waiter_overlap__overlap__open_probs,actor waiter overlap,overlap,small,open,True,open_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,False,underdetermined,...,probability,0.15,NaN,5,open,True,True,1,True,True
4,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,well_posed,0,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,True,implausible,...,mc_choice,NaN,D,5,mc_full,True,False,0,True,False


In [42]:
df.columns

Index(['example_id', 'vignette_name', 'problem_type', 'intersection_size',
       'response_type', 'has_statistics', 'variant', 'prompt', 'well_posed',
       'normative', 'p_c_and_d_given_a', 'normative_choice',
       'normative_percent', 'normative_open', 'confidence_required',
       'numeric_score_percent', 'numeric_score_choice', 'scepticism_required',
       'scepticism_score_target', 'option_a_label', 'option_b_label',
       'option_c_label', 'option_d_label', 'option_e_label', 'option_a_lure',
       'option_b_lure', 'option_c_lure', 'option_d_lure', 'option_e_lure',
       'option_f_label', 'option_g_label', 'option_h_label', 'option_f_lure',
       'option_g_lure', 'option_h_lure', 'model', 'llm_response', 'reasoning',
       'answer_line', 'confidence_line', 'parsed_answer_type',
       'parsed_percent', 'parsed_choice', 'parsed_confidence', 'scoring_type',
       'parseable', 'score', 'score_value', 'parseable_bool', 'score_true'],
      dtype='str')

In [43]:
df['problem_type'].value_counts()

problem_type
well_posed    20
overlap       14
Name: count, dtype: int64

In [44]:
  df['variant'].value_counts()

variant
mc_full_probs       20
mc_numeric_probs     7
open_probs           7
Name: count, dtype: int64

In [45]:
 df['intersection_size'].value_counts()

intersection_size
0         20
small      8
large      4
medium     2
Name: count, dtype: int64

## `mc_numeric_probs` detail

For each vignette: MC options A–E, the model's letter (`parsed_choice`), partition shortcut letter (`numeric_score_choice`), scepticism fields, and score.

In [46]:
MC_NUMERIC_CHOICE_COLS = [f"option_{letter}_label" for letter in "abcde"]


def format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter, col in zip("ABCDE", MC_NUMERIC_CHOICE_COLS):
        value = row.get(col)
        if pd.notna(value) and str(value).strip():
            parts.append(f"{letter}: {value}")
    return " | ".join(parts)


mc_numeric_probs = df[df["variant"] == "mc_numeric_probs"].copy()
mc_numeric_probs["choices_offered"] = mc_numeric_probs.apply(format_mc_choices, axis=1)

mc_numeric_probs_view = mc_numeric_probs[
    [
        "vignette_name",
        "choices_offered",
        "parsed_choice",
        "numeric_score_choice",
        "scepticism_required",
        "scepticism_score_target",
        "score",
        "score_value",
        "normative_choice",
        "answer_line",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 140)
mc_numeric_probs_view

,vignette_name,choices_offered,parsed_choice,numeric_score_choice,scepticism_required,scepticism_score_target,score,score_value,normative_choice,answer_line
6,CA Trump voter,A: About 10% | B: About 1% | C: About 4% | D: About 6% | E: About 13%,A,A,False,NaN,True,1,A,A
2,actor waiter overlap,A: About 0% | B: About 19%,B,A,False,NaN,False,0,A,B
12,covid vaccine (blue/red),A: About 20% | B: About 8% | C: About 14% | D: About 0% | E: About 27%,C,A,False,NaN,False,0,A,C
18,discharged weapon (last year),A: About 91% | B: About 0% | C: About 70% | D: About 89% | E: About 44%,A,A,False,NaN,True,1,A,A
24,healthcare employment,A: About 88% | B: About 40% | C: About 26% | D: About 87% | E: About 11%,A,A,False,NaN,True,1,A,A
28,military overseas (federal pool),A: About 82% | B: About 73% | C: About 36% | D: About 63% | E: About 40%,A,A,False,NaN,True,1,A,A
32,professional drivers speeding,A: About 3% | B: About 0% | C: About 2% | D: About 10%,D,A,False,NaN,False,0,A,D


## `open_probs` detail

Benchmark scoring fields plus re-parsed response values. Each `llm_response` is re-parsed (strip trailing confidence, extract all % / 0–1 decimals). **`score_true`** is whether any candidate is within ±0.5 pp of **`scepticism_score_target`**.

In [47]:
import sys
from pathlib import Path

if "ROOT" not in globals():
    ROOT = Path.cwd()
    if not (ROOT / "data" / "base_rate").is_dir():
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks.base_rate import (
    load_benchmark,
    matches_scepticism_target,
    parse_open_response,
    strip_trailing_confidence,
)

benchmark_items = load_benchmark()


def rescore_open_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_open_response(str(row["llm_response"]))
    body, confidence = strip_trailing_confidence(str(row["llm_response"]))
    score_true = matches_scepticism_target(item, parsed)
    return pd.Series(
        {
            "response_body": body,
            "parsed_confidence": confidence,
            "parsed_numbers": list(parsed.percent_candidates),
            "parsed_percent_rescored": parsed.percent,
            "parsed_answer_type_rescored": parsed.answer_type,
            "parseable_rescored": parsed.answer_type != "unparseable",
            "score_true": score_true,
        }
    )


open_probs = df[df["variant"] == "open_probs"].copy()
open_probs = open_probs.drop(columns=["score_true"], errors="ignore")
open_probs = pd.concat([open_probs, open_probs.apply(rescore_open_row, axis=1)], axis=1)

# Write rescored values back into the main frame for open_probs rows.
idx = open_probs.index
df.loc[idx, "parsed_percent"] = pd.to_numeric(open_probs["parsed_percent_rescored"], errors="coerce")
df.loc[idx, "parsed_answer_type"] = open_probs["parsed_answer_type_rescored"].astype("string")
df.loc[idx, "parseable"] = open_probs["parseable_rescored"].astype(bool)
df.loc[idx, "parseable_bool"] = open_probs["parseable_rescored"].astype(bool)
df.loc[idx, "score"] = open_probs["score_true"].astype(bool)
df.loc[idx, "score_value"] = open_probs["score_true"].astype(int)
df.loc[idx, "score_true"] = open_probs["score_true"].astype(bool)

OPEN_PROBS_SCORING_COLUMNS = [
    "example_id",
    "vignette_name",
    "normative",
    "scepticism_required",
    "normative_percent",
    "normative_open",
    "numeric_score_percent",
    "scepticism_score_target",
    "parsed_numbers",
    "parsed_percent_rescored",
    "parsed_answer_type_rescored",
    "parsed_confidence",
    "parseable_rescored",
    "score_true",
]

open_probs_view = (
    open_probs[OPEN_PROBS_SCORING_COLUMNS]
    .sort_values(["vignette_name", "normative"])
    .reset_index(drop=True)
)

print(
    "Rescored open_probs score_true:",
    int(open_probs["score_true"].sum()),
    "/",
    len(open_probs),
)
pd.set_option("display.max_colwidth", 120)
open_probs_view

Rescored open_probs score_true: 1 / 7


,example_id,vignette_name,normative,scepticism_required,normative_percent,normative_open,numeric_score_percent,scepticism_score_target,parsed_numbers,parsed_percent_rescored,parsed_answer_type_rescored,parsed_confidence,parsed_confidence,parseable_rescored,score_true
0,ca_trump_voter__open_probs,CA Trump voter,well_posed,False,9.91800,9.9%,9.91200,9.918,[21.1],21.10,probability,5,5,True,False
1,actor_waiter_overlap__overlap__open_probs,actor waiter overlap,underdetermined,False,0.03737,0.04%,0.03966,0.03737,"[0.15, 15.0]",0.15,probability,5,5,True,True
2,covid_vaccine_blue_red__open_probs,covid vaccine (blue/red),well_posed,False,19.62000,20%,19.62000,19.62,[30.0],30.00,probability,5,5,True,False
3,discharged_weapon_last_year__open_probs,discharged weapon (last year),well_posed,False,91.21000,91%,91.21000,91.21,[99.7],99.70,probability,5,5,True,False
4,healthcare_employment__open_probs,healthcare employment,well_posed,False,88.01000,88%,88.01000,88.01,[97.8],97.80,probability,5,5,True,False
5,military_overseas_federal_pool__open_probs,military overseas (federal pool),well_posed,False,81.54000,82%,81.52000,81.54,[95.9],95.90,probability,5,5,True,False
6,professional_drivers_speeding__overlap__open_probs,professional drivers speeding,underdetermined,False,2.64800,2.6%,2.65800,2.648,[5.6],5.60,probability,5,5,True,False


## `open_probs` vs `mc_numeric_probs` vs normative

Side-by-side for the **7 canonical** vignettes (both variants present). **Normative** = overlap-aware P(A|T) (`normative_percent`). **Partition** = partition-shortcut P(A|T) (`numeric_score_percent`; MC numeric score target is `n/a`, pass = normative letter or same rounded %).

In [48]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks.base_rate import load_benchmark, parse_open_response, parse_response

benchmark_items = load_benchmark()
items_meta = pd.read_csv(ROOT / "data" / "base_rate" / "items.csv")


def _label_percent(label: str) -> float | None:
    text = (label or "").strip()
    if not text.startswith("About "):
        return None
    try:
        return float(text.removeprefix("About ").removesuffix("%"))
    except ValueError:
        return None


def _format_mc_menu(row: pd.Series) -> str:
    parts = []
    for letter in "ABCDE":
        label = row.get(f"option_{letter}_label")
        if pd.notna(label) and str(label).strip():
            parts.append(f"{letter}:{label}")
    return " | ".join(parts)


comparison_rows: list[dict] = []
canonical_vignettes = sorted(
    items_meta.loc[
        ~items_meta["scepticism_required"].astype(str).str.lower().eq("true"),
        "vignette_name",
    ].unique()
)

for vignette_name in canonical_vignettes:
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = item_mc.get(f"option_{mc_choice.lower()}_label", "") if mc_choice else ""
    mc_pct = _label_percent(str(mc_label))

    normative_pct = float(item_open["normative_percent"])
    partition_pct = float(item_mc["numeric_score_percent"])
    open_pct = parsed_open.percent

    comparison_rows.append(
        {
            "vignette_name": vignette_name,
            "normative_open": item_open["normative_open"],
            "normative_pct": normative_pct,
            "partition_pct": partition_pct,
            "normative_mc_letter": item_mc["normative_choice"],
            "mc_menu": _format_mc_menu(item_mc),
            "open_parsed_pct": open_pct,
            "open_delta_vs_norm_pp": None if open_pct is None else open_pct - normative_pct,
            "open_score": bool(open_row.get("score_true", open_row.get("score_value", 0))),
            "mc_choice": mc_choice,
            "mc_label": mc_label,
            "mc_parsed_pct": mc_pct,
            "mc_delta_vs_norm_pp": None if mc_pct is None else mc_pct - normative_pct,
            "mc_delta_vs_partition_pp": None if mc_pct is None else mc_pct - partition_pct,
            "mc_score": bool(mc_row.get("score_true", mc_row.get("score_value", 0))),
            "same_norm_partition_rounded": round(normative_pct) == round(partition_pct),
        }
    )

open_vs_mc = pd.DataFrame(comparison_rows).sort_values("vignette_name")

print(
    "open_probs pass:",
    int(open_vs_mc["open_score"].sum()),
    "/",
    len(open_vs_mc),
    "| mc_numeric_probs pass:",
    int(open_vs_mc["mc_score"].sum()),
    "/",
    len(open_vs_mc),
)

display(
    open_vs_mc[
        [
            "vignette_name",
            "normative_open",
            "normative_pct",
            "partition_pct",
            "open_parsed_pct",
            "open_delta_vs_norm_pp",
            "open_score",
            "mc_choice",
            "mc_label",
            "mc_parsed_pct",
            "mc_delta_vs_norm_pp",
            "mc_score",
            "normative_mc_letter",
        ]
    ]
)

with pd.option_context("display.max_colwidth", None):
    display(open_vs_mc[["vignette_name", "mc_menu"]])

open_probs pass: 1 / 7 | mc_numeric_probs pass: 4 / 7


,vignette_name,normative_open,normative_pct,partition_pct,open_parsed_pct,open_delta_vs_norm_pp,open_score,mc_choice,mc_label,mc_parsed_pct,mc_delta_vs_norm_pp,mc_score,normative_mc_letter
0,CA Trump voter,9.9%,9.91800,9.91200,21.10,11.18200,False,A,About 10%,10.0,0.08200,True,A
1,actor waiter overlap,0.04%,0.03737,0.03966,0.15,0.11263,True,B,About 19%,19.0,18.96263,False,A
2,covid vaccine (blue/red),20%,19.62000,19.62000,30.00,10.38000,False,C,About 14%,14.0,-5.62000,False,A
3,discharged weapon (last year),91%,91.21000,91.21000,99.70,8.49000,False,A,About 91%,91.0,-0.21000,True,A
4,healthcare employment,88%,88.01000,88.01000,97.80,9.79000,False,A,About 88%,88.0,-0.01000,True,A
5,military overseas (federal pool),82%,81.54000,81.52000,95.90,14.36000,False,A,About 82%,82.0,0.46000,True,A
6,professional drivers speeding,2.6%,2.64800,2.65800,5.60,2.95200,False,D,About 10%,10.0,7.35200,False,A


,vignette_name,mc_menu
0,CA Trump voter,
1,actor waiter overlap,
2,covid vaccine (blue/red),
3,discharged weapon (last year),
4,healthcare employment,
5,military overseas (federal pool),
6,professional drivers speeding,


### Printable comparison (canonical vignettes)

Per vignette: source probabilities (vignette CSVs via `load_vignettes`; `items.csv` stores `p_c_and_d_given_a` only), normative / open / MC answers, numeric MC options, and full `mc_numeric_probs` prompt.

In [49]:
import re

from benchmarks.base_rate import parse_open_response, parse_response
from scripts.build_base_rate_prompts import load_vignettes

items_meta = pd.read_csv(ROOT / "data" / "base_rate" / "items.csv")
benchmark_df = pd.read_csv(ROOT / "data" / "base_rate" / "benchmark.csv")

canonical_vignettes = sorted(
    items_meta.loc[
        ~items_meta["scepticism_required"].astype(str).str.lower().eq("true"),
        "vignette_name",
    ].unique()
)

vignette_by_name = {
    v.name: v
    for v in load_vignettes()
    if v.normative != "implausible"
}


def mc_numeric_options_prompt(prompt: str) -> str:
    """Extract the A–E option block from an mc_numeric_probs prompt."""
    lines = [line.strip() for line in prompt.splitlines() if line.strip()]
    option_lines = [line for line in lines if re.match(r"^[A-E]\.\s", line)]
    return " | ".join(option_lines)


def format_source_ps(v, item_row: pd.Series) -> str:
    parts = [
        f"P(A)={v.p_a:.6g}",
        f"P(C|A)={v.q_c:.6g}",
        f"P(D|A)={v.q_d:.6g}",
        f"P(T|C)={v.s_c:.6g}",
        f"P(T|D)={v.s_d:.6g}",
        f"P(T|N)={v.f_n:.6g}",
    ]
    p_cd_items = item_row.get("p_c_and_d_given_a")
    if pd.notna(p_cd_items):
        parts.append(f"P(C∩D|A)={float(p_cd_items):.6g} (items.csv)")
    elif v.p_cd:
        parts.append(f"P(C∩D|A)={v.p_cd:.6g}")
    return " | ".join(parts)


print_rows: list[dict] = []
for vignette_name in canonical_vignettes:
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]
    bench_row = benchmark_df.loc[benchmark_df["example_id"] == mc_row["example_id"]].iloc[0]
    vignette = vignette_by_name[vignette_name]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = (
        str(item_mc.get(f"option_{mc_choice.lower()}_label", ""))
        if mc_choice
        else ""
    )
    full_prompt = str(bench_row["prompt"])

    print_rows.append(
        {
            "vignette_name": vignette_name,
            "source_ps": format_source_ps(vignette, item_open),
            "p_a": vignette.p_a,
            "p_c_given_a": vignette.q_c,
            "p_d_given_a": vignette.q_d,
            "p_t_given_c": vignette.s_c,
            "p_t_given_d": vignette.s_d,
            "p_t_given_n": vignette.f_n,
            "p_c_and_d_given_a": float(item_open["p_c_and_d_given_a"]),
            "normative_pct": float(item_open["normative_percent"]),
            "open_parsed_pct": parsed_open.percent,
            "mc_label": f"{mc_choice} {mc_label}".strip(),
            "numeric_prompt": mc_numeric_options_prompt(full_prompt),
            "prompt": full_prompt,
        }
    )

print_table = pd.DataFrame(print_rows).sort_values("vignette_name")

print(f"{'vignette_name':<32} {'normative':>10} {'open':>10} {'MC label':>14}")
print("-" * 72)
for row in print_table.itertuples(index=False):
    open_pct = "—" if pd.isna(row.open_parsed_pct) else f"{row.open_parsed_pct:.4g}%"
    print(f"\n{row.vignette_name}")
    print(f"  source Ps:   {row.source_ps}")
    print(f"  normative:   {row.normative_pct:.4g}%")
    print(f"  open parsed: {open_pct}")
    print(f"  MC label:    {row.mc_label}")
    print(f"  numeric prompt: {row.numeric_prompt}")
    print("  prompt:")
    for line in row.prompt.splitlines():
        print(f"    {line}")

print_table.drop(columns=["prompt"])

vignette_name                     normative       open       MC label
------------------------------------------------------------------------

CA Trump voter
  source Ps:   P(A)=0.13 | P(C|A)=0.38 | P(D|A)=0.6 | P(T|C)=0.31 | P(T|D)=0.27 | P(T|N)=0.38 | P(C∩D|A)=0 (items.csv)
  normative:   9.918%
  open parsed: 21.1%
  MC label:    A About 10%
  numeric prompt: A. About 10% | B. About 1% | C. About 4% | D. About 6% | E. About 13%
  prompt:
    You are a statistical consultant. Your task is to estimate a conditional probability from the information below.
    
    Among US registered voters, 13% are registered in California and the remainder are registered elsewhere.
    Among those registered in California, 60% registered in southern California and 38% registered in other parts of the state.
    Among those registered in southern California, 27% voted for Donald Trump in the 2024 presidential election; among those registered in other parts of the state, 31% voted for Donald Trump in 

,vignette_name,source_ps,p_a,p_c_given_a,p_d_given_a,p_t_given_c,p_t_given_d,p_t_given_n,p_c_and_d_given_a,normative_pct,open_parsed_pct,mc_label,numeric_prompt
0,CA Trump voter,P(A)=0.13 | P(C|A)=0.38 | P(D|A)=0.6 | P(T|C)=0.31 | P(T|D)=0.27 | P(T|N)=0.38 | P(C∩D|A)=0 (items.csv),0.13000,0.380,0.600,0.310,0.270,0.3800,0.000,9.91800,21.10,A About 10%,A. About 10% | B. About 1% | C. About 4% | D. About 6% | E. About 13%
1,actor waiter overlap,P(A)=0.00034 | P(C|A)=0.035 | P(D|A)=0.298 | P(T|C)=0.65 | P(T|D)=0.55 | P(T|N)=0.16 | P(C∩D|A)=0.018 (items.csv),0.00034,0.035,0.298,0.650,0.550,0.1600,0.018,0.03737,0.15,B About 19%,A. About 0% | B. About 19%
2,covid vaccine (blue/red),P(A)=0.27 | P(C|A)=0.71 | P(D|A)=0.29 | P(T|C)=0.08 | P(T|D)=0.1 | P(T|N)=0.13 | P(C∩D|A)=0 (items.csv),0.27000,0.710,0.290,0.080,0.100,0.1300,0.000,19.62000,30.00,C About 14%,A. About 20% | B. About 8% | C. About 14% | D. About 0% | E. About 27%
3,discharged weapon (last year),P(A)=0.44 | P(C|A)=0.68 | P(D|A)=0.3 | P(T|C)=0.003 | P(T|D)=0.002 | P(T|N)=0.0002 | P(C∩D|A)=0 (items.csv),0.44000,0.680,0.300,0.003,0.002,0.0002,0.000,91.21000,99.70,A About 91%,A. About 91% | B. About 0% | C. About 70% | D. About 89% | E. About 44%
4,healthcare employment,P(A)=0.11 | P(C|A)=0.1 | P(D|A)=0.9 | P(T|C)=0.54 | P(T|D)=0.6 | P(T|N)=0.01 | P(C∩D|A)=0 (items.csv),0.11000,0.100,0.900,0.540,0.600,0.0100,0.000,88.01000,97.80,A About 88%,A. About 88% | B. About 40% | C. About 26% | D. About 87% | E. About 11%
5,military overseas (federal pool),P(A)=0.4 | P(C|A)=0.35 | P(D|A)=0.51 | P(T|C)=0.58 | P(T|D)=0.64 | P(T|N)=0.08 | P(C∩D|A)=0 (items.csv),0.40000,0.350,0.510,0.580,0.640,0.0800,0.000,81.54000,95.90,A About 82%,A. About 82% | B. About 73% | C. About 36% | D. About 63% | E. About 40%
6,professional drivers speeding,P(A)=0.024 | P(C|A)=0.534 | P(D|A)=0.145 | P(T|C)=0.16 | P(T|D)=0.1 | P(T|N)=0.09 | P(C∩D|A)=0.003 (items.csv),0.02400,0.534,0.145,0.160,0.100,0.0900,0.003,2.64800,5.60,D About 10%,A. About 3% | B. About 0% | C. About 2% | D. About 10%


In [50]:
open_probs_responses = (
    open_probs[
        ["vignette_name", "normative_open", "parsed_numbers", "llm_response"]
    ]
    .sort_values("vignette_name")
    .reset_index(drop=True)
)

with pd.option_context("display.max_colwidth", None, "display.width", None):
    display(open_probs_responses)

,vignette_name,normative_open,parsed_numbers,llm_response
0,CA Trump voter,9.9%,[21.1],21.1%\n5
1,actor waiter overlap,0.04%,"[0.15, 15.0]",0.15%\n5
2,covid vaccine (blue/red),20%,[30.0],30.0%\n5
3,discharged weapon (last year),91%,[99.7],99.7%\n5
4,healthcare employment,88%,[97.8],97.8%\n5
5,military overseas (federal pool),82%,[95.9],95.9%\n5
6,professional drivers speeding,2.6%,[5.6],5.6%\n5


## `mc_full_probs` detail

Benchmark scoring fields plus re-parsed MC choice (bottom-up scan for A–H), **only rows with `scepticism_required=true`**. **`score`** is whether **`parsed_choice_rescored`** matches **`scepticism_score_target`** (single letter, or any of `F|G|H`).

In [51]:
from benchmarks.base_rate import (
    load_benchmark,
    matches_scepticism_target,
    parse_response,
)

if "benchmark_items" not in globals():
    benchmark_items = load_benchmark()


def rescore_mc_full_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_response(str(row["llm_response"]), scoring_type="mc_full")
    score_true = matches_scepticism_target(item, parsed)
    return pd.Series(
        {
            "answer_line_rescored": parsed.answer_line,
            "parsed_confidence": parsed.confidence,
            "parsed_choice_rescored": parsed.choice,
            "parsed_answer_type_rescored": parsed.answer_type,
            "parseable_rescored": parsed.choice is not None,
            "score": score_true,
        }
    )


mc_full_probs = df[df["variant"] == "mc_full_probs"].copy()
mc_full_probs = mc_full_probs.drop(columns=["score"], errors="ignore")
mc_full_probs = pd.concat(
    [mc_full_probs, mc_full_probs.apply(rescore_mc_full_row, axis=1)],
    axis=1,
)

MC_FULL_PROBS_SCORING_COLUMNS = [
    "example_id",
    "vignette_name",
    "normative",
    "scepticism_required",
    "normative_choice",
    "numeric_score_choice",
    "scepticism_score_target",
    "parsed_choice_rescored",
    "parsed_answer_type_rescored",
    "answer_line_rescored",
    "parsed_confidence",
    "parseable_rescored",
    "score",
]

mc_full_probs_view = (
    mc_full_probs.loc[
        mc_full_probs["scepticism_required"].astype(str).str.lower().eq("true"),
        MC_FULL_PROBS_SCORING_COLUMNS,
    ]
    .sort_values(["vignette_name", "normative"])
    .reset_index(drop=True)
)

print(
    "mc_full_probs score (scepticism_required):",
    int(mc_full_probs_view["score"].sum()),
    "/",
    len(mc_full_probs_view),
)

pd.set_option("display.max_colwidth", 120)
mc_full_probs_view.head(n=5)

mc_full_probs score (scepticism_required): 0 / 13


,example_id,vignette_name,normative,scepticism_required,normative_choice,numeric_score_choice,scepticism_score_target,parsed_choice_rescored,parsed_answer_type_rescored,answer_line_rescored,parsed_confidence,parsed_confidence,parseable_rescored,score
0,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,implausible,True,A,A,H,D,mc_choice,D,5,5,True,False
1,actor_waiter_overlap__overlap__implausible__mc_full_probs,actor waiter overlap,implausible,True,A,A,H,A,mc_choice,A,5,5,True,False
2,college_stem_work__overlap__implausible__mc_full_probs,college STEM work,implausible,True,A,B,H,C,mc_choice,C,5,5,True,False
3,college_stem_work__overlap__mc_full_probs,college STEM work,underdetermined,True,A,B,F|G|H,A,mc_choice,A,5,5,True,False
4,covid_vaccine_blue_red__implausible__mc_full_probs,covid vaccine (blue/red),implausible,True,A,A,H,D,mc_choice,D,5,5,True,False


In [52]:
df['reasoning'].value_counts(), df['confidence_required'].value_counts(), df['normative_choice'].value_counts()

(Series([], Name: count, dtype: int64),
 confidence_required
 True    34
 Name: count, dtype: int64,
 normative_choice
 A    27
 Name: count, dtype: int64)

## Scores by `response_type`

In [53]:
RESPONSE_TYPE_ORDER = ["open", "mc_numeric", "mc_full"]


def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, parseability mix, and mean score for each group value."""
    work = df.copy()
    if "parseable_bool" not in work.columns:
        work["parseable_bool"] = True
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "score_rate": grouped["score_value"].mean(),
        }
    )
    summary["score_pct"] = (summary["score_rate"] * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])

    return summary


by_response_type = score_summary_table("response_type", order=RESPONSE_TYPE_ORDER)
by_response_type

,n,score_true,score_false,unparseable,score_rate,score_pct
response_type,,,,,,
open,7,1,6,0,0.142857,14.3
mc_numeric,7,4,3,0,0.571429,57.1
mc_full,20,4,16,0,0.200000,20.0


## Scores by `variant`

In [54]:
VARIANT_ORDER = [
    "open_probs",
    "mc_numeric_probs",
    "mc_full_probs",
]

by_variant = score_summary_table("variant", order=VARIANT_ORDER)
by_variant

,n,score_true,score_false,unparseable,score_rate,score_pct
variant,,,,,,
open_probs,7,1,6,0,0.142857,14.3
mc_numeric_probs,7,4,3,0,0.571429,57.1
mc_full_probs,20,4,16,0,0.200000,20.0


## Scores by `vignette_name`

In [55]:
by_vignette = score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)
by_vignette

,n,score_true,score_false,unparseable,score_rate,score_pct
vignette_name,,,,,,
CA Trump voter,4,1,3,0,0.25,25.0
actor waiter overlap,4,2,2,0,0.50,50.0
college STEM work,2,0,2,0,0.00,0.0
covid vaccine (blue/red),4,0,4,0,0.00,0.0
diabetes insulin obese,2,0,2,0,0.00,0.0
discharged weapon (last year),4,2,2,0,0.50,50.0
english teacher humanities,2,0,2,0,0.00,0.0
healthcare employment,4,1,3,0,0.25,25.0
military overseas (federal pool),4,2,2,0,0.50,50.0


## Optional: split by model when multiple LLMs are present

In [56]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "response_type"], observed=True)["score_value"]
        .mean()
        .unstack("response_type")
        .reindex(columns=RESPONSE_TYPE_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")

Single model in file — see tables above.


In [57]:
df['model'].value_counts()

model
google/gemini-2.5-flash    34
Name: count, dtype: int64

In [58]:
Trump = df.query("vignette_name == 'CA Trump voter'")
Trump

,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,parsed_answer_type,parsed_percent,parsed_choice,parsed_confidence,scoring_type,parseable,score,score_value,parseable_bool,score_true
4,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,well_posed,0,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,True,implausible,...,mc_choice,NaN,D,5,mc_full,True,False,0,True,False
5,ca_trump_voter__mc_full_probs,CA Trump voter,well_posed,0,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,True,well_posed,...,mc_choice,NaN,C,5,mc_full,True,False,0,True,False
6,ca_trump_voter__mc_numeric_probs,CA Trump voter,well_posed,0,mc_numeric,True,mc_numeric_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,True,well_posed,...,mc_choice,NaN,A,5,mc_numeric,True,True,1,True,True
7,ca_trump_voter__open_probs,CA Trump voter,well_posed,0,open,True,open_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,True,well_posed,...,probability,21.1,NaN,5,open,True,False,0,True,False
